# Grokking: Weight Averaging & Distillation ExperimentReproduces the grokking experiment (Power et al., 2022) on modular division (mod 97), then compares:- **Baseline**: single model trained on full data- **4 Specialists**: each trained on 25% of the data- **Weight Averaged**: average specialist weights, fine-tune on full data- **Distilled**: knowledge distillation from specialists into a fresh studentAll methods use the **same total compute budget** for fair comparison.

In [ ]:
!pip install -q pytorch_lightning mod sympy blobfile

## Clone Repo

In [ ]:
import os, sysREPO_DIR = '/kaggle/working/grok'if not os.path.exists(REPO_DIR):    !git clone https://github.com/openai/grok {REPO_DIR}sys.path.insert(0, REPO_DIR)os.chdir(REPO_DIR)print(f'Repo at: {REPO_DIR}')

## Apply FixesThe original code never logs val_loss/val_accuracy/full_train_acc in Lightning 2.x. Also adds `split_n_ways` (data splitting) which is missing from the original repo.

In [ ]:
import reTRAINING_FILE = os.path.join(REPO_DIR, 'grok', 'training.py')with open(TRAINING_FILE, 'r') as f:    code = f.read()# Fix 1: validation_step - append {} to _val_step_outputs on skipold_val_step = (    '        if self.next_epoch_to_eval < self.current_epoch:\n'    '            self.next_epoch_to_eval = self.current_epoch\n'    '        if self.current_epoch != self.next_epoch_to_eval:\n'    '            return {}')new_val_step = (    '        if self.next_epoch_to_eval < self.current_epoch:\n'    '            self.next_epoch_to_eval = self.current_epoch\n'    '        if self.current_epoch != self.next_epoch_to_eval:\n'    '            self._val_step_outputs.append({})\n'    '            return {}')assert old_val_step in code, 'validation_step pattern not found'code = code.replace(old_val_step, new_val_step)print('Fixed validation_step')# Fix 2: on_validation_epoch_end - remove stale next_epoch_to_eval updateold_val_end = (    '        if validation_is_real:\n'    '            self.next_epoch_to_eval = max(\n'    '                int(1.02 * self.next_epoch_to_eval), self.next_epoch_to_eval + 1\n'    '            )')assert old_val_end in code, 'on_validation_epoch_end header not found'code = code.replace(old_val_end, '        if validation_is_real:')print('Fixed on_validation_epoch_end')# Fix 3: checkpoint_path fallbackold_ckpt = (    '            self.trainer.save_checkpoint(\n'    '                os.path.join(\n'    '                    self.hparams.checkpoint_path,\n'    '                    "epoch_" + str(self.current_epoch) + ".ckpt",\n'    '                )\n'    '            )')new_ckpt = (    '            ckpt_path = getattr(self.hparams, "checkpoint_path", None) or os.path.join(\n'    '                self.hparams.logdir, "checkpoints"\n'    '            )\n'    '            os.makedirs(ckpt_path, exist_ok=True)\n'    '            self.trainer.save_checkpoint(\n'    '                os.path.join(\n'    '                    ckpt_path,\n'    '                    "epoch_" + str(self.current_epoch) + ".ckpt",\n'    '                )\n'    '            )')assert old_ckpt in code, 'checkpoint_path pattern not found'code = code.replace(old_ckpt, new_ckpt)print('Fixed checkpoint_path')with open(TRAINING_FILE, 'w') as f:    f.write(code)# Add split_n_ways to data.py (not in original OpenAI repo)DATA_FILE = os.path.join(REPO_DIR, 'grok', 'data.py')with open(DATA_FILE, 'r') as f:    data_code = f.read()if 'split_n_ways' not in data_code:    split_n_ways_code = (        '\n'        '    @classmethod\n'        '    def split_n_ways(\n'        '        cls,\n'        '        dataset: "ArithmeticDataset",\n'        '        n: int,\n'        '        seed: int = 42,\n'        '    ) -> List["ArithmeticDataset"]:\n'        '        """Partition an existing dataset into n disjoint shards."""\n'        '        gen = torch.Generator()\n'        '        gen.manual_seed(seed)\n'        '        perm = torch.randperm(len(dataset.data), generator=gen)\n'        '        chunks = torch.chunk(perm, n)\n'        '        result = []\n'        '        for chunk in chunks:\n'        '            shard = cls.__new__(cls)\n'        '            shard.tokenizer = dataset.tokenizer\n'        '            shard.name = dataset.name\n'        '            shard.train = True\n'        '            shard.data = dataset.data[chunk]\n'        '            result.append(shard)\n'        '        return result\n'    )    marker = '    @classmethod\n    def _make_lists'    idx = data_code.find(marker)    if idx != -1:        data_code = data_code[:idx] + split_n_ways_code + '\n\n' + data_code[idx:]    with open(DATA_FILE, 'w') as f:        f.write(data_code)    print('Added split_n_ways to data.py')else:    print('split_n_ways already in data.py')# Fix multi_training.py importsMULTI_FILE = os.path.join(REPO_DIR, 'grok', 'multi_training.py')with open(MULTI_FILE, 'r') as f:    multi_code = f.read()if 'from grok.data import ArithmeticDataset' in multi_code and 'ArithmeticIterator' not in multi_code:    multi_code = multi_code.replace(        'from grok.data import ArithmeticDataset',        'from grok.data import ArithmeticDataset, ArithmeticIterator'    )    with open(MULTI_FILE, 'w') as f:        f.write(multi_code)    print('Added ArithmeticIterator import')else:    print('multi_training.py imports already correct')print('All fixes applied!')

## Imports & GPU Detection

In [ ]:
import torchimport copyimport jsonimport numpy as npimport pandas as pdimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltfrom argparse import Namespacefrom grok.multi_training import (    train_multi_with_distillation,    add_multi_args,)GPU_ID = 0 if torch.cuda.is_available() else -1DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'print(f'Device: {DEVICE}')print(f'GPU ID: {GPU_ID}')

## Experiment ConfigurationPaper-exact hyperparameters for modular division (mod 97).**Compute budget:** 100k total steps.- 4 specialists x 25k steps each = 100k total specialist compute- Merged model: 100k steps (fine-tune from averaged weights)- Distillation student: 100k steps (train from scratch)- Baseline: 100k steps (train from scratch)For a quick test, reduce `FINAL_STEPS` to 1000-5000.

In [ ]:
# --- Experiment parameters ---FINAL_STEPS = 100000        # Total compute budget (reduce for quick test: e.g. 2000)N_MODELS = 4                # Number of specialist modelsSPECIALIST_STEPS = FINAL_STEPS // N_MODELS  # Equal compute per specialistDISTILL_STEPS = FINAL_STEPS # Same budget as baselineRANDOM_SEED = 42# --- Paths ---LOGDIR = '/kaggle/working/logs'DATADIR = os.path.join(REPO_DIR, 'data')EXPERIMENT_NAME = 'grokking_experiment'print(f'Final steps (baseline/merged/distilled): {FINAL_STEPS}')print(f'Specialist steps: {SPECIALIST_STEPS} each x {N_MODELS} = {SPECIALIST_STEPS * N_MODELS} total')print(f'Distill steps: {DISTILL_STEPS}')print(f'Logdir: {LOGDIR}')

## Build Hyperparameters

In [ ]:
parser = add_multi_args()hparams = parser.parse_args([])  # empty args, use defaults below# Override with our confighparams.logdir = LOGDIRhparams.datadir = DATADIRhparams.math_operator = '/'hparams.train_data_pct = 50hparams.n_models = N_MODELShparams.specialist_steps = SPECIALIST_STEPShparams.final_steps = FINAL_STEPShparams.distill_steps = DISTILL_STEPShparams.distill_temperature = 2.0hparams.distill_alpha = 0.5hparams.experiment_name = EXPERIMENT_NAMEhparams.run_baseline = Truehparams.random_seed = RANDOM_SEEDhparams.gpu = GPU_IDhparams.weight_decay = 1hparams.max_lr = 1e-3hparams.warmup_steps = 10hparams.n_layers = 2hparams.n_heads = 4hparams.d_model = 128hparams.dropout = 0.0hparams.max_context_len = 50hparams.non_linearity = 'relu'hparams.batchsize = 0hparams.weight_noise = 0.0hparams.noise_factor = 0hparams.weight_decay_kind = 'to_zero'hparams.anneal_lr = Falsehparams.anneal_lr_steps = 100000hparams.save_activations = Falsehparams.save_outputs = Falsehparams.operand_length = Noneprint(hparams)print(f'Model: {hparams.n_layers}L-{hparams.n_heads}H-{hparams.d_model}D (~455K params)')print(f'Operator: {hparams.math_operator} mod 97 | Train: {hparams.train_data_pct}%')

## Run Full ExperimentPhases:1. Train 4 specialists on disjoint data shards2. Average specialist weights3. Fine-tune merged model on full data4. Distill knowledge from specialists into a fresh student5. Train baseline (random init) on full data

In [ ]:
experiment_dir = train_multi_with_distillation(hparams)print(f'Experiment dir: {experiment_dir}')

## Discover All Metrics Files

In [ ]:
def find_metrics_csv(experiment_dir, subdir):    base = os.path.join(experiment_dir, subdir)    if not os.path.exists(base):        return None    lightning_dir = os.path.join(base, 'lightning_logs')    if not os.path.exists(lightning_dir):        return None    versions = sorted([d for d in os.listdir(lightning_dir) if d.startswith('version_')])    for v in reversed(versions):        csv_path = os.path.join(lightning_dir, v, 'metrics.csv')        if os.path.exists(csv_path):            return csv_path    return Nonemetrics_paths = {}for i in range(N_MODELS):    p = find_metrics_csv(experiment_dir, f'specialist_{i}')    if p:        metrics_paths[f'specialist_{i}'] = pfor name in ['merged_average', 'baseline']:    p = find_metrics_csv(experiment_dir, name)    if p:        metrics_paths[name] = pdistill_csv = os.path.join(experiment_dir, 'distilled', 'distill_metrics.csv')if os.path.exists(distill_csv):    metrics_paths['distilled'] = distill_csvdistill_val_csv = os.path.join(experiment_dir, 'distilled', 'distill_val_metrics.csv')if os.path.exists(distill_val_csv):    metrics_paths['distilled_val'] = distill_val_csvprint('Found metrics:')for name, path in metrics_paths.items():    print(f'  {name}: {path}')

## Load & Inspect All Metrics

In [ ]:
all_data = {}for name, path in metrics_paths.items():    df = pd.read_csv(path)    all_data[name] = df    non_empty = {col: df[col].notna().sum() for col in df.columns}    print(f'{name}: {len(df)} rows')    for col in ['train_loss', 'train_accuracy', 'val_loss', 'val_accuracy', 'full_train_acc', 'full_train_loss', 'student_acc']:        if col in non_empty:            print(f'  {col}: {non_empty[col]} values')

## Plot: Accuracy Curves (Grokking Curve)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))colors = {'baseline': '#2196F3', 'merged_average': '#4CAF50', 'distilled': '#9C27B0'}labels = {'baseline': 'Baseline', 'merged_average': 'Weight Averaged', 'distilled': 'Distilled'}# Left: Validation Accuracyax = axes[0]for i in range(N_MODELS):    name = f'specialist_{i}'    if name in all_data:        df = all_data[name]        if 'val_accuracy' in df.columns:            d = df.dropna(subset=['val_accuracy'])            ax.plot(d['step'], d['val_accuracy'], alpha=0.3, linewidth=0.8, color='gray')for name in ['baseline', 'merged_average']:    if name in all_data:        df = all_data[name]        if 'val_accuracy' in df.columns:            d = df.dropna(subset=['val_accuracy'])            ax.plot(d['step'], d['val_accuracy'], label=labels[name], linewidth=2, color=colors[name])if 'distilled_val' in all_data:    df = all_data['distilled_val']    if 'val_accuracy' in df.columns:        ax.plot(df['step'], df['val_accuracy'], label='Distilled', linewidth=2, color=colors['distilled'])ax.set_title('Validation Accuracy', fontsize=14)ax.set_xlabel('Step')ax.set_ylabel('Accuracy (%)')ax.legend(loc='lower right')ax.grid(True, alpha=0.3)ax.set_ylim(0, 105)# Right: Training Accuracyax = axes[1]for i in range(N_MODELS):    name = f'specialist_{i}'    if name in all_data:        df = all_data[name]        col = 'full_train_acc' if 'full_train_acc' in df.columns else 'train_accuracy'        if col in df.columns:            d = df.dropna(subset=[col])            ax.plot(d['step'], d[col], alpha=0.3, linewidth=0.8, color='gray')for name in ['baseline', 'merged_average']:    if name in all_data:        df = all_data[name]        col = 'full_train_acc' if 'full_train_acc' in df.columns else 'train_accuracy'        if col in df.columns:            d = df.dropna(subset=[col])            ax.plot(d['step'], d[col], label=labels[name], linewidth=2, color=colors[name])if 'distilled' in all_data:    df = all_data['distilled']    if 'student_acc' in df.columns:        ax.plot(df['step'], df['student_acc'], label='Distilled', linewidth=2, color=colors['distilled'], alpha=0.7)ax.set_title('Training Accuracy', fontsize=14)ax.set_xlabel('Step')ax.set_ylabel('Accuracy (%)')ax.legend(loc='lower right')ax.grid(True, alpha=0.3)ax.set_ylim(0, 105)fig.suptitle('Grokking: Division mod 97 - Accuracy', fontsize=16, fontweight='bold')plt.tight_layout()plt.savefig('/kaggle/working/accuracy_curves.png', dpi=150, bbox_inches='tight')plt.show()print('Saved: /kaggle/working/accuracy_curves.png')

## Plot: Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))# Left: Validation Lossax = axes[0]for i in range(N_MODELS):    name = f'specialist_{i}'    if name in all_data:        df = all_data[name]        if 'val_loss' in df.columns:            d = df.dropna(subset=['val_loss'])            ax.plot(d['step'], d['val_loss'], alpha=0.3, linewidth=0.8, color='gray')for name in ['baseline', 'merged_average']:    if name in all_data:        df = all_data[name]        if 'val_loss' in df.columns:            d = df.dropna(subset=['val_loss'])            ax.plot(d['step'], d['val_loss'], label=labels[name], linewidth=2, color=colors[name])ax.set_title('Validation Loss', fontsize=14)ax.set_xlabel('Step')ax.set_ylabel('Loss')ax.legend()ax.grid(True, alpha=0.3)ax.set_yscale('log')# Right: Training Lossax = axes[1]for i in range(N_MODELS):    name = f'specialist_{i}'    if name in all_data:        df = all_data[name]        col = 'full_train_loss' if 'full_train_loss' in df.columns else 'train_loss'        if col in df.columns:            d = df.dropna(subset=[col])            ax.plot(d['step'], d[col], alpha=0.3, linewidth=0.8, color='gray')for name in ['baseline', 'merged_average']:    if name in all_data:        df = all_data[name]        col = 'full_train_loss' if 'full_train_loss' in df.columns else 'train_loss'        if col in df.columns:            d = df.dropna(subset=[col])            ax.plot(d['step'], d[col], label=labels[name], linewidth=2, color=colors[name])if 'distilled' in all_data:    df = all_data['distilled']    if 'loss' in df.columns:        window = max(1, len(df) // 200)        smoothed = df['loss'].rolling(window=window, center=True).mean()        ax.plot(df['step'], smoothed, label='Distilled', linewidth=2, color=colors['distilled'])ax.set_title('Training Loss', fontsize=14)ax.set_xlabel('Step')ax.set_ylabel('Loss')ax.legend()ax.grid(True, alpha=0.3)ax.set_yscale('log')fig.suptitle('Grokking: Division mod 97 - Loss', fontsize=16, fontweight='bold')plt.tight_layout()plt.savefig('/kaggle/working/loss_curves.png', dpi=150, bbox_inches='tight')plt.show()print('Saved: /kaggle/working/loss_curves.png')

## Plot: Combined Summary Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))# Top Left: Val Accuracyax = axes[0, 0]for i in range(N_MODELS):    name = f'specialist_{i}'    if name in all_data:        df = all_data[name]        if 'val_accuracy' in df.columns:            d = df.dropna(subset=['val_accuracy'])            ax.plot(d['step'], d['val_accuracy'], alpha=0.3, linewidth=0.8, color='gray')for name in ['baseline', 'merged_average']:    if name in all_data:        df = all_data[name]        if 'val_accuracy' in df.columns:            d = df.dropna(subset=['val_accuracy'])            ax.plot(d['step'], d['val_accuracy'], label=labels[name], linewidth=2, color=colors[name])if 'distilled_val' in all_data:    df = all_data['distilled_val']    ax.plot(df['step'], df['val_accuracy'], label='Distilled', linewidth=2, color=colors['distilled'])ax.set_title('Validation Accuracy', fontsize=13)ax.set_xlabel('Step')ax.set_ylabel('Accuracy (%)')ax.legend(loc='lower right')ax.grid(True, alpha=0.3)ax.set_ylim(0, 105)# Top Right: Val Loss (log)ax = axes[0, 1]for i in range(N_MODELS):    name = f'specialist_{i}'    if name in all_data:        df = all_data[name]        if 'val_loss' in df.columns:            d = df.dropna(subset=['val_loss'])            ax.plot(d['step'], d['val_loss'], alpha=0.3, linewidth=0.8, color='gray')for name in ['baseline', 'merged_average']:    if name in all_data:        df = all_data[name]        if 'val_loss' in df.columns:            d = df.dropna(subset=['val_loss'])            ax.plot(d['step'], d['val_loss'], label=labels[name], linewidth=2, color=colors[name])ax.set_title('Validation Loss (log scale)', fontsize=13)ax.set_xlabel('Step')ax.set_ylabel('Loss')ax.set_yscale('log')ax.legend()ax.grid(True, alpha=0.3)# Bottom Left: Train Accuracyax = axes[1, 0]for i in range(N_MODELS):    name = f'specialist_{i}'    if name in all_data:        df = all_data[name]        col = 'full_train_acc' if 'full_train_acc' in df.columns else 'train_accuracy'        if col in df.columns:            d = df.dropna(subset=[col])            ax.plot(d['step'], d[col], alpha=0.3, linewidth=0.8, color='gray')for name in ['baseline', 'merged_average']:    if name in all_data:        df = all_data[name]        col = 'full_train_acc' if 'full_train_acc' in df.columns else 'train_accuracy'        if col in df.columns:            d = df.dropna(subset=[col])            ax.plot(d['step'], d[col], label=labels[name], linewidth=2, color=colors[name])if 'distilled' in all_data:    df = all_data['distilled']    if 'student_acc' in df.columns:        ax.plot(df['step'], df['student_acc'], label='Distilled', linewidth=2, color=colors['distilled'], alpha=0.7)ax.set_title('Training Accuracy', fontsize=13)ax.set_xlabel('Step')ax.set_ylabel('Accuracy (%)')ax.legend(loc='lower right')ax.grid(True, alpha=0.3)ax.set_ylim(0, 105)# Bottom Right: Final Accuracy Bar Chartax = axes[1, 1]method_names = []final_accs = []bar_colors = []for name, label_text in [('baseline', 'Baseline'), ('merged_average', 'Weight Avg'), ('distilled', 'Distilled')]:    if name == 'distilled' and 'distilled_val' in all_data:        df = all_data['distilled_val']        if 'val_accuracy' in df.columns and len(df) > 0:            method_names.append(label_text)            final_accs.append(float(df['val_accuracy'].iloc[-1]))            bar_colors.append(colors.get(name, 'gray'))    elif name in all_data:        df = all_data[name]        if 'val_accuracy' in df.columns:            d = df.dropna(subset=['val_accuracy'])            if len(d) > 0:                method_names.append(label_text)                final_accs.append(float(d['val_accuracy'].iloc[-1]))                bar_colors.append(colors.get(name, 'gray'))if method_names:    bars = ax.bar(method_names, final_accs, color=bar_colors, alpha=0.8, edgecolor='black', linewidth=1.2)    for bar, acc in zip(bars, final_accs):        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,                f'{acc:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')    ax.set_ylim(0, max(final_accs) * 1.3 if final_accs else 100)ax.set_title('Final Validation Accuracy', fontsize=13)ax.set_ylabel('Accuracy (%)')ax.grid(True, alpha=0.3, axis='y')fig.suptitle(    f'Grokking Experiment: {hparams.math_operator} mod 97 | '    f'{N_MODELS} specialists | {FINAL_STEPS:,} steps budget',    fontsize=16, fontweight='bold')plt.tight_layout()plt.savefig('/kaggle/working/grokking_dashboard.png', dpi=150, bbox_inches='tight')plt.show()print('Saved: /kaggle/working/grokking_dashboard.png')

## Final Results Summary

In [ ]:
print('=' * 60)print('FINAL RESULTS')print('=' * 60)for name, label_text in [('baseline', 'Baseline'), ('merged_average', 'Weight Averaged'), ('distilled', 'Distilled')]:    if name == 'distilled' and 'distilled_val' in all_data:        df = all_data['distilled_val']        if 'val_accuracy' in df.columns and len(df) > 0:            print(f'  {label_text:20s}: val_acc = {float(df["val_accuracy"].iloc[-1]):.2f}%')    elif name in all_data:        df = all_data[name]        if 'val_accuracy' in df.columns:            d = df.dropna(subset=['val_accuracy'])            if len(d) > 0:                print(f'  {label_text:20s}: val_acc = {float(d["val_accuracy"].iloc[-1]):.2f}%')print(f'Specialists:')for i in range(N_MODELS):    name = f'specialist_{i}'    if name in all_data:        df = all_data[name]        if 'val_accuracy' in df.columns:            d = df.dropna(subset=['val_accuracy'])            if len(d) > 0:                print(f'    specialist_{i}: val_acc = {float(d["val_accuracy"].iloc[-1]):.2f}%')print(f'Plots saved to /kaggle/working/')print(f'  - grokking_dashboard.png (combined)')print(f'  - accuracy_curves.png')print(f'  - loss_curves.png')